In [ ]:
from google.cloud import aiplatform

# Set your project ID and location
project_id = "your-project-id"
location = "us-central1"

# Create a Vertex AI client
client = aiplatform.gapic.JobServiceClient(client_options={"api_endpoint": f"{location}-aiplatform.googleapis.com"})

# Define the input data path
input_data_path = "gs://your-bucket/reddit.jsonl"

# Define the output directory
output_dir = "gs://your-bucket/output"

# Define the training script
training_script = "train.py"

# Define the training job name
job_name = "reddit-training-job"

# Define the training job spec
training_job_spec = {
    "display_name": job_name,
    "job_spec": {
        "worker_pool_specs": [
            {
                "machine_spec": {
                    "machine_type": "n1-standard-4",
                    "accelerator_type": "NVIDIA_TESLA_K80",
                    "accelerator_count": 1,
                },
                "replica_count": 1,
                "python_package_spec": {
                    "executor_image_uri": "gcr.io/cloud-aiplatform/training/tf-gpu.2-6:latest",
                    "package_uris": ["gs://your-bucket/train_package.tar.gz"],
                    "python_module": "train",
                    "args": ["--input_data", input_data_path, "--output_dir", output_dir],
                },
            }
        ],
        "base_output_directory": output_dir,
    },
}

# Submit the training job
response = client.create_custom_job(parent=f"projects/{project_id}/locations/{location}", custom_job=training_job_spec)

# Get the training job ID
job_id = response.name.split("/")[-1]

print(f"Training job submitted: {job_id}")